In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
train_df = pd.read_csv("../data/train_rewrite_001.csv")

In [5]:
import citation_utils

In [6]:
import random

def gen_sample_for_query(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)
    cc_with_score_l = dense_index.search_with_score(query, 1000)
    cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)

    TOPK=50
    
    # 计算有gold的cc
    cc_contains_gold = set()
    for cc, score in cc_with_score_l[:TOPK]:
        for gold in gold_citation_set:
            if gold in cc['text']:
                cc_contains_gold.add(cc['citation'])

    # 算出所有的evidence，evidence会记录它属于哪个cc
    evidence_d = {}
    for cc, score in cc_with_score_l[:TOPK]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        for citation, first_appear_idx in parsed_cc['citations']:
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            evidence_d[evidence_text] = {"cc_citation": cc['citation']}

    positive_evidence_set = set()
    for evidence_text, _ in evidence_d.items():
        for gold_citation in gold_citation_l:
            if gold_citation in evidence_text:
                positive_evidence_set.add(evidence_text)

    hard_negative_evidence_set = set()
    for evidence_text, meta in evidence_d.items():
        if evidence_text in positive_evidence_set:
            continue

        if meta["cc_citation"] in cc_contains_gold:
            continue
            
        hard_negative_evidence_set.add(evidence_text)
        
    pos_cnt = len(positive_evidence_set)
    # 4 个hard negative
    hard_negative_evidence_list = list(hard_negative_evidence_set)
    random.shuffle(hard_negative_evidence_list)

    pos_l = list(positive_evidence_set)
    neg_l = hard_negative_evidence_list[:pos_cnt*4]

    rand_l = []
    # 2 个random negative
    l1 = cc_with_score_l[:500].copy()
    random.shuffle(l1)
    for cc, score in l1:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        if len(parsed_cc['citations']) > 0:
            citation, first_appear_idx = parsed_cc['citations'][0]
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            rand_l.append(evidence_text)
        if len(rand_l) >= pos_cnt*2:
            break
            

    return pos_l, neg_l, rand_l
    
    # print(query_id, 'positive_evidence_set.len:', len(positive_evidence_set), 
    #       "hard_negative_evidence_set.len:", len(hard_negative_evidence_set),
    #       "gold.len:", len(gold_citation_set))

In [7]:
import random

def gen_sample_for_query2(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)
    cc_with_score_l = dense_index.search_with_score(query, 1000)
    cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)

    TOPK=5
    
    # 计算有gold的cc
    cc_contains_gold = set()
    for cc, score in cc_with_score_l[:TOPK]:
        for gold in gold_citation_set:
            if gold in cc['text']:
                cc_contains_gold.add(cc['citation'])

    # 算出TOPK的evidence，evidence会记录它属于哪个cc
    evidence_d = {}
    for cc, score in cc_with_score_l[:TOPK]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        for citation, first_appear_idx in parsed_cc['citations']:
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            evidence_d[evidence_text] = {"cc_citation": cc['citation']}

    positive_evidence_set = set()
    for evidence_text, _ in evidence_d.items():
        for gold_citation in gold_citation_l:
            if gold_citation in evidence_text:
                positive_evidence_set.add(evidence_text)

    # TOPK to TOPK + 30，排除掉有gold的cc
    evidence_d = {}
    for cc, score in cc_with_score_l[TOPK:TOPK+15]:
        has_gold = False
        for gold in gold_citation_set:
            if gold in cc['text']:
                has_gold = True
                break
        if has_gold:
            continue #排除掉有gold的cc
        else:
            parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
            for citation, first_appear_idx in parsed_cc['citations']:
                evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
                evidence_d[evidence_text] = {"cc_citation": cc['citation']}

    hard_negative_evidence_set = set()
    for evidence_text, meta in evidence_d.items():
        if evidence_text in positive_evidence_set:
            continue

        if meta["cc_citation"] in cc_contains_gold:
            continue
            
        hard_negative_evidence_set.add(evidence_text)
        
    pos_cnt = len(positive_evidence_set)
    # 4 个hard negative
    hard_negative_evidence_list = list(hard_negative_evidence_set)
    random.shuffle(hard_negative_evidence_list)

    pos_l = list(positive_evidence_set)
    neg_l = hard_negative_evidence_list[:pos_cnt*5]

    rand_l = []
    # 2 个random negative
    l1 = cc_with_score_l[:500].copy()
    random.shuffle(l1)
    for cc, score in l1:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        if len(parsed_cc['citations']) > 0:
            citation, first_appear_idx = parsed_cc['citations'][0]
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            rand_l.append(evidence_text)
        if len(rand_l) >= pos_cnt*6:
            break
            

    return pos_l, neg_l, rand_l
    
    # print(query_id, 'positive_evidence_set.len:', len(positive_evidence_set), 
    #       "hard_negative_evidence_set.len:", len(hard_negative_evidence_set),
    #       "gold.len:", len(gold_citation_set))

In [8]:

def gen_sample_for_query_neg_before_pos(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)

    cc_with_score_l = dense_index.search_with_score(query, 1000)

    sorted_cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)
    
    first_positive_idx = 0

    for idx, (cc, score) in enumerate(sorted_cc_with_score_l):
        found_first_positive_idx = len(sorted_cc_with_score_l)
        for gold in gold_citation_set:
            if gold in cc['text']:
                found_first_positive_idx = idx
                break
        if found_first_positive_idx < len(sorted_cc_with_score_l):
            break

    hard_neg_l = sorted_cc_with_score_l[:found_first_positive_idx]

    if found_first_positive_idx >= len(sorted_cc_with_score_l):
        return [], [], []
        
    pos = sorted_cc_with_score_l[found_first_positive_idx]

    pos_l, neg_l, rand_l = [], [], []


    parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(pos[0]['text'])
    for idx, sentence in enumerate(parsed_cc['sentences']):
        for gold in gold_citation_set:
            if gold in sentence:
                evidence = citation_utils.build_evidence(parsed_cc['sentences'], idx, 5)
                pos_l.append(evidence)
                break # 第一个含gold的句子

    for cc, score in hard_neg_l[:5]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        citations = parsed_cc['citations']
        for citation in citations:
            for idx, sentence in enumerate(parsed_cc['sentences']):
                if citation in sentence:
                    neg_l.append(citation_utils.build_evidence(parsed_cc['sentences'], idx, 5))
                    break
    random.shuffle(neg_l)
    neg_l = neg_l[:len(pos_l)*5]

    return pos_l, neg_l, rand_l
            
            
        

In [25]:


def get_cc_list(cc_list, idx, window_size=3):
    half = window_size // 2
    left = max(0, idx - half)
    right = min(len(cc_list), idx + half + 1)
    return cc_list[left:right].copy()
    
def gen_sample_for_query_neg_in_pos_window(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)

    window_size = 3

    cc_with_score_l = dense_index.search_with_score(query, 1000)

    sorted_cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)
    
    pos_idx_l = []
    
    for idx, (cc, score) in enumerate(sorted_cc_with_score_l):
        cc['gold'] = False
        for gold in gold_citation_set:
            if gold in cc['text']:
                pos_idx_l.append(idx)
                cc['gold'] = True
                break

    pos_l = []
    
    for pos in [sorted_cc_with_score_l[pos_idx] for pos_idx in pos_idx_l]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(pos[0]['text'])
        for idx, sentence in enumerate(parsed_cc['sentences']):
            for gold in gold_citation_set:
                if gold in sentence:
                    evidence = citation_utils.build_evidence(parsed_cc['sentences'], idx, 5)
                    pos_l.append(evidence)
                    break # 第一个含gold的句子

    tmp_l = []
    
    for pos_idx in pos_idx_l:
        l0 = get_cc_list(sorted_cc_with_score_l, pos_idx, window_size)
        l1 = [cc for cc,score in l0 if cc['gold'] is False]
        tmp_l.extend(l1)

    hard_neg_l = []
    seen_set = set()
    for cc in tmp_l:
        if cc['citation'] in seen_set:
            continue
        else:
            seen_set.add(cc['citation'])
            hard_neg_l.append(cc)

    neg_l = []
    for cc in hard_neg_l:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        citations = parsed_cc['citations']
        for citation,_ in citations:
            for idx, sentence in enumerate(parsed_cc['sentences']):
                if citation in sentence:
                    neg_l.append(citation_utils.build_evidence(parsed_cc['sentences'], idx, 5))
                    break
    random.shuffle(neg_l)
    neg_l = neg_l[:len(pos_l)*5]

    return pos_l, neg_l, []
            
            
        

In [35]:
def contain_gold_citation(text, gold_citation_set):
    for gold in gold_citation_set:
        if gold in text:
            return True
    return False

In [38]:
def gen_sample_for_query_neg_in_search_by_evidence(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)

    cc_with_score_l = dense_index.search_with_score(query, 100)

    sorted_cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)
    
    pos_idx_l = []

    pos_cc_set = set()
    for idx, (cc, score) in enumerate(sorted_cc_with_score_l):
        cc['gold'] = False
        for gold in gold_citation_set:
            if gold in cc['text']:
                pos_idx_l.append(idx)
                cc['gold'] = True
                pos_cc_set.add(cc['citation'])
                break


    pos_l = []

    for pos in [sorted_cc_with_score_l[pos_idx] for pos_idx in pos_idx_l]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(pos[0]['text'])
        for idx, sentence in enumerate(parsed_cc['sentences']):
            for gold in gold_citation_set:
                if gold in sentence:
                    evidence = citation_utils.build_evidence(parsed_cc['sentences'], idx, 5)
                    pos_l.append(evidence)
                    break # 第一个含gold的句子

    tmp_l = []
    for evidence in pos_l:
        _l = dense_index.search_with_score(evidence, 5)
        neg_cc_with_score_l = [cc for cc,score in _l if cc['citation'] not in pos_cc_set]
        for neg_cc in neg_cc_with_score_l:
            if contain_gold_citation(neg_cc['text'], gold_citation_set):
                pass
            else:
                tmp_l.append(neg_cc)

    hard_neg_l = []
    seen_set = set()
    for cc in tmp_l:
        if cc['citation'] in seen_set:
            continue
        else:
            seen_set.add(cc['citation'])
            hard_neg_l.append(cc)
            
    neg_l = []
    for cc in hard_neg_l:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        citations = parsed_cc['citations']
        for citation,_ in citations:
            for idx, sentence in enumerate(parsed_cc['sentences']):
                if citation in sentence:
                    neg_l.append(citation_utils.build_evidence(parsed_cc['sentences'], idx, 5))
                    break
    random.shuffle(neg_l)
    neg_l = neg_l[:len(pos_l)*5]

    return pos_l, neg_l, []

In [39]:
import json
all_sample = []
for query_id, query, gold_citations in zip(train_df['query_id'], train_df['query2'], train_df['gold_citations']):
    pos_l, neg_l, rand_l = gen_sample_for_query_neg_in_search_by_evidence(query_id, query, gold_citations.split(';'))
    if len(pos_l) > 0:
        print(query_id, ", pos_l.len:", len(pos_l), ", neg_l.len:", len(neg_l), ", rand_l.len:", len(rand_l))
    for evidence in pos_l:
        all_sample.append({"query":query, "passage": evidence, "label":1})
    for evidence in neg_l:
        all_sample.append({"query":query, "passage": evidence, "label":0})
    # for evidence in rand_l:
    #    all_sample.append({"query":query, "passage": evidence, "label":"0"})

random.shuffle(all_sample)

train_size = int(len(all_sample)*0.9)

train = all_sample[:train_size]
valid = all_sample[train_size:]

with open("../ft_data/train.jsonl", "w+", encoding="utf-8") as of:
    for sample in train:
        of.write(json.dumps(sample, ensure_ascii=False)+"\n")

with open("../ft_data/valid.jsonl", "w+", encoding="utf-8") as of:
    for sample in valid:
        of.write(json.dumps(sample, ensure_ascii=False)+"\n")
      
    

train_0001 , pos_l.len: 16 , neg_l.len: 64 , rand_l.len: 0
train_0002 , pos_l.len: 15 , neg_l.len: 75 , rand_l.len: 0
train_0004 , pos_l.len: 12 , neg_l.len: 60 , rand_l.len: 0
train_0005 , pos_l.len: 26 , neg_l.len: 49 , rand_l.len: 0
train_0006 , pos_l.len: 71 , neg_l.len: 57 , rand_l.len: 0
train_0007 , pos_l.len: 46 , neg_l.len: 130 , rand_l.len: 0
train_0009 , pos_l.len: 62 , neg_l.len: 125 , rand_l.len: 0
train_0010 , pos_l.len: 32 , neg_l.len: 67 , rand_l.len: 0
train_0014 , pos_l.len: 86 , neg_l.len: 315 , rand_l.len: 0
train_0015 , pos_l.len: 1 , neg_l.len: 5 , rand_l.len: 0
train_0016 , pos_l.len: 28 , neg_l.len: 106 , rand_l.len: 0
train_0018 , pos_l.len: 27 , neg_l.len: 82 , rand_l.len: 0
train_0020 , pos_l.len: 4 , neg_l.len: 20 , rand_l.len: 0
train_0022 , pos_l.len: 3 , neg_l.len: 15 , rand_l.len: 0
train_0023 , pos_l.len: 11 , neg_l.len: 42 , rand_l.len: 0
train_0024 , pos_l.len: 1 , neg_l.len: 3 , rand_l.len: 0
train_0025 , pos_l.len: 11 , neg_l.len: 55 , rand_l.len: 0

In [21]:
import json
import random
from collections import defaultdict

train_l = []
with open("../ft_data/train.jsonl", encoding="utf-8") as inf:
    for line in inf:
        train_l.append(json.loads(line.strip()))

train_d = defaultdict(list)
for d in train_l:
    train_d[d['query']].append(d)

for query, l in train_d.items():
    for d in l:
        if d['label'] == '1':
            d['label'] = 1
        elif d['label'] == '0':
            d['label'] = 0

train_pos_cnt_d = defaultdict(list)
for query, l in train_d.items():
    count = 0
    for d in l:
        if d['label'] == 1:
            count += 1
    train_pos_cnt_d[query] = count

all_l = []
all_pos_l = []
for query, l in train_d.items():
    for d in l:
        all_l.append((query,d))
        if d['label'] == 1:
            all_pos_l.append((query,d))
        
print("all_l.len:", len(all_l))
print("all_pos_l.len:", len(all_pos_l))

train_rand_d = defaultdict(list)

for query, l in train_d.items():
    rand_cnt = train_pos_cnt_d[query] * 2
    rand_l = []
    random.shuffle(all_l)
    # print("===>rand_cnt:", rand_cnt)
    idx = 0
    while len(rand_l) < rand_cnt:
        q,d = all_l[idx]
        if q == query:
            idx += 1
            continue
        else:
            d1 = d.copy()
            d1['query'] = query
            d1['label'] = 0
            rand_l.append(d1)
            idx += 1
    train_rand_d[query]=rand_l.copy()

all_rand_l = []
for query, l in train_rand_d.items():
    for d in l:
        all_rand_l.append(d)
print("all_rand_l.len:", len(all_rand_l))

for query, l in train_d.items():
    l.extend(train_rand_d[query])

train_d_l = []
for _,l in train_d.items():
    for d in l:
        train_d_l.append(d)
        
random.shuffle(train_d_l)
print("train_d_l.len:", len(train_d_l))


with open("../ft_data/train_with_rand.jsonl", "w+", encoding="utf-8") as of:
    for sample in train_d_l:
        of.write(json.dumps(sample, ensure_ascii=False)+"\n")


all_l.len: 59696
all_pos_l.len: 13930
all_rand_l.len: 27860
train_d_l.len: 87556
